# 01 — Arrival Data Preparation (V9.0)

**加载、清洗、统一 Jan-Oct 2025 航班数据（148K arrivals）**

输出：`data/processed/arrivals_processed.parquet` + 辅助数据（departures、missed approaches、FAA events）

In [1]:
# Standard imports
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

# === Project paths (RCF: only change BASE_DIR) ===
BASE_DIR = Path('../../..')             # RCF: Path('/path/to/project')
DATA_RAW = BASE_DIR / 'data' / 'raw' / 'LGA_Dataset'
DATA_PROCESSED = BASE_DIR / 'data' / 'processed'

print(f"Data directory: {DATA_RAW}")
print(f"Output directory: {DATA_PROCESSED}")

Data directory: ../../../data/raw/LGA_Dataset
Output directory: ../../../data/processed


## 1. Arrival Flight Data (5 CSVs)

Three schema variants across CSV files:
- **Old** (May-June, July, Aug-Sept): `Marketing Airline Desc`, `Body Type Desc`, `Total Calculated Delay`, etc.
- **New v1** (previous truncated files, no longer used): `Airline Name` → full name
- **New v2** (Jan-May, Sept-Oct full data): `Carrier` → 3-letter ICAO code (AAL, DAL, RPA, etc.)

`harmonize_flight_columns()` detects schema and renames to canonical (old) names.
For v2 schema, carrier codes are mapped to marketing airline names using a lookup
built from overlapping May data between old and new files.

In [2]:
# === Carrier code → Marketing Airline name mapping ===
# Built from overlapping May data between old files (Marketing Airline Desc)
# and new files (Carrier code as Call Sign prefix).
# Regional carriers mapped to their marketing brand at LGA.
CARRIER_TO_AIRLINE = {
    # --- Mainline ---
    'AAL': 'American Airlines',
    'DAL': 'Delta Air Lines',
    'UAL': 'United Airlines',
    'SWA': 'Southwest Airlines',
    'FFT': 'Frontier Airlines',
    'JBU': 'JetBlue Airways',
    'NKS': 'Spirit Airlines',
    'ACA': 'Air Canada',
    'POE': 'Porter Airlines',
    'GXA': 'GlobalX Airlines',
    # --- Regional (operating as mainline brand at LGA) ---
    'RPA': 'American Airlines',   # Republic Airways → American Eagle
    'ASH': 'American Airlines',   # Mesa Airlines → American Eagle
    'EDV': 'Delta Air Lines',     # Endeavor Air → Delta Connection
    'SKW': 'United Airlines',     # SkyWest → United Express
    'GJS': 'United Airlines',     # GoJet → United Express
    'JZA': 'Air Canada',          # Jazz Aviation → Air Canada Express
    # --- General Aviation / Charter ---
    'EJA': 'General Aviation',    # NetJets
    'LXJ': 'General Aviation',    # Flexjet
    'EJM': 'General Aviation',    # Executive Jet Mgmt
    'GPD': 'General Aviation',
    'CNS': 'General Aviation',
    'VJA': 'General Aviation',
    'TWY': 'General Aviation',
    'ASP': 'General Aviation',
    'BOG': 'General Aviation',
    'COL': 'General Aviation',
    'WUP': 'General Aviation',
    'JRE': 'General Aviation',
    'XFL': 'General Aviation',
}

# Column mapping: new v2 schema → canonical (old) schema
# v2 files have: Carrier, Body Type, Delay, Movement Time, Ramp Time,
#   total_taxi_time (lowercase!), Gate Occupancy Time, Runway, Terminal+Terminal Code
NEW_V2_TO_CANONICAL = {
    'Body Type': 'Body Type Desc',
    'Delay': 'Total Calculated Delay',
    'Movement Time': 'Total  Movement Time',       # old has double space
    'Ramp Time': 'Total Ramp Time',
    'total_taxi_time': 'Total Taxi Time Calc',      # lowercase in v2!
    'Gate Occupancy Time': 'Total Gate Occ Time',
}

# Column mapping: new v1 schema → canonical (kept for backward compat)
NEW_V1_TO_CANONICAL = {
    'Airline Name': 'Marketing Airline Desc',
    'Body Type': 'Body Type Desc',
    'Delay': 'Total Calculated Delay',
    'Movement Time': 'Total  Movement Time',
    'Taxi Time': 'Total Taxi Time Calc',
}

def harmonize_flight_columns(df, direction='arrival'):
    """Detect schema variant and rename to canonical (old) column names.
    
    Handles 3 schemas:
    - Old (May-Sept): already canonical, no changes needed
    - New v1 (had 'Airline Name'): simple rename
    - New v2 (has 'Carrier' code): rename + carrier code → airline name lookup
    """
    if 'Carrier' in df.columns:
        # === New v2 schema (Jan-May, Sept-Oct full data) ===
        rename_map = NEW_V2_TO_CANONICAL.copy()
        
        # Runway: single 'Runway' column → direction-specific
        if 'Runway' in df.columns:
            rename_map['Runway'] = 'Arrival Runway' if direction == 'arrival' else 'Departure Runway'
        
        df = df.rename(columns=rename_map)
        
        # Map carrier code → marketing airline name
        df['Marketing Airline Desc'] = df['Carrier'].map(CARRIER_TO_AIRLINE).fillna('General Aviation')
        
        # Convert 'NULL' strings to NaN in Body Type
        if 'Body Type Desc' in df.columns:
            df['Body Type Desc'] = df['Body Type Desc'].replace('NULL', np.nan)
        
        # Drop redundant Terminal column (keep Terminal Code)
        if 'Terminal' in df.columns and 'Terminal Code' in df.columns:
            df = df.drop(columns=['Terminal'])
        
        # Report mapping stats
        n_mapped = df['Marketing Airline Desc'].ne('General Aviation').sum()
        n_ga = (df['Marketing Airline Desc'] == 'General Aviation').sum()
        unmapped = df.loc[
            (df['Marketing Airline Desc'] == 'General Aviation') & (~df['Carrier'].isin(CARRIER_TO_AIRLINE)),
            'Carrier'
        ].unique()
        unmapped_str = f", unmapped codes: {list(unmapped[:5])}" if len(unmapped) > 0 else ""
        print(f"  → v2 schema: Carrier codes mapped ({n_mapped:,} commercial, {n_ga:,} GA/other{unmapped_str})")
    
    elif 'Airline Name' in df.columns:
        # === New v1 schema (previous truncated files) ===
        rename_map = NEW_V1_TO_CANONICAL.copy()
        if 'Runway' in df.columns:
            rename_map['Runway'] = 'Arrival Runway' if direction == 'arrival' else 'Departure Runway'
        df = df.rename(columns=rename_map)
        if 'Body Type Desc' in df.columns:
            df['Body Type Desc'] = df['Body Type Desc'].replace('NULL', np.nan)
        if 'Terminal' in df.columns and 'Terminal Code' in df.columns:
            df = df.drop(columns=['Terminal'])
        print(f"  → v1 schema: Airline Name renamed to Marketing Airline Desc")
    
    else:
        # === Old schema (May-Sept) ===
        print(f"  → Old schema, no renaming needed")
    
    return df

print("harmonize_flight_columns() defined.")
print(f"Carrier codes: {len(CARRIER_TO_AIRLINE)} mapped ({sum(1 for v in CARRIER_TO_AIRLINE.values() if v != 'General Aviation')} commercial)")
print(f"v2 column mappings: {list(NEW_V2_TO_CANONICAL.keys())}")

harmonize_flight_columns() defined.
Carrier codes: 29 mapped (16 commercial)
v2 column mappings: ['Body Type', 'Delay', 'Movement Time', 'Ramp Time', 'total_taxi_time', 'Gate Occupancy Time']


In [3]:
# Load all arrival CSV files (5 files, Jan-Oct 2025)
arrival_files = [
    'LGA_Jan-May2025_Arrivals.csv',       # v2 schema (Carrier code), Jan 1 - May 22
    'LGA_May-June2025_Arrivals.csv',       # old schema, May 22 - Jun 30
    'LGA_July2025_Arrivals.csv',           # old schema, Jul 1 - Jul 31
    'LGA_Aug-Sept2025_Arrivals.csv',       # old schema, Aug 1 - Sep 1
    'LGA_Sept-Oct2025_Arrivals.csv',       # v2 schema (Carrier code), Sep 2 - Oct 31
]

dfs = []
for file in arrival_files:
    filepath = DATA_RAW / file
    if filepath.exists():
        df_temp = pd.read_csv(filepath)
        print(f"Loaded {file}: {len(df_temp):,} rows")
        df_temp = harmonize_flight_columns(df_temp, direction='arrival')
        dfs.append(df_temp)
    else:
        print(f"WARNING: {file} not found — skipping")

# Combine all data
df = pd.concat(dfs, ignore_index=True)
print(f"\nCombined: {len(df):,} rows")

# Sort by datetime and deduplicate
df['_sort_dt'] = pd.to_datetime(df['Block Schedule'], errors='coerce')
df = df.sort_values('_sort_dt').drop(columns=['_sort_dt'])
before_dedup = len(df)
df = df.drop_duplicates(
    subset=['Registration', 'Call Sign', 'Block Schedule', 'Block Actual'],
    keep='first'
).reset_index(drop=True)
print(f"Dedup: {before_dedup:,} → {len(df):,} ({before_dedup - len(df):,} duplicates removed)")
print(f"Total arrival records: {len(df):,}")

# Verify airline distribution
print(f"\nAirline distribution (top 10):")
print(df['Marketing Airline Desc'].value_counts().head(10).to_string())

Loaded LGA_Jan-May2025_Arrivals.csv: 67,997 rows
  → v2 schema: Carrier codes mapped (66,643 commercial, 1,354 GA/other, unmapped codes: ['WMN', 'Unknown', 'JNY', 'FXV', 'RVF'])
Loaded LGA_May-June2025_Arrivals.csv: 20,244 rows
  → Old schema, no renaming needed
Loaded LGA_July2025_Arrivals.csv: 16,039 rows
  → Old schema, no renaming needed
Loaded LGA_Aug-Sept2025_Arrivals.csv: 16,415 rows
  → Old schema, no renaming needed
Loaded LGA_Sept-Oct2025_Arrivals.csv: 30,141 rows
  → v2 schema: Carrier codes mapped (29,486 commercial, 655 GA/other, unmapped codes: ['Unknown', 'FXV', 'TMP', 'IJM', 'HER'])

Combined: 150,836 rows
Dedup: 150,836 → 148,165 (2,671 duplicates removed)
Total arrival records: 148,165

Airline distribution (top 10):
Marketing Airline Desc
Delta Air Lines       57003
American Airlines     52991
Southwest Airlines     9570
United Airlines        9548
Spirit Airlines        4827
Air Canada             4719
JetBlue Airways        3890
General Aviation       2201
Frontier

## 2. Departure Data (for lag features)

LGA departure data for computing `lga_dep_delay_1h` lag features. Expanded to 5 files.

In [4]:
# Load all departure CSV files (5 files, Jan-Oct 2025)
departure_files = [
    'LGA_Jan-May2025_Departures.csv',       # v2 schema
    'LGA_May-June2025_Departures.csv',       # old schema
    'LGA_July2025_Departures.csv',           # old schema
    'LGA_Aug-Sept2025_Departures.csv',       # old schema
    'LGA_Sept-Oct2025_Departures.csv',       # v2 schema
]

dep_dfs = []
for file in departure_files:
    filepath = DATA_RAW / file
    if filepath.exists():
        df_temp = pd.read_csv(filepath)
        print(f"Loaded {file}: {len(df_temp):,} rows")
        df_temp = harmonize_flight_columns(df_temp, direction='departure')
        dep_dfs.append(df_temp)
    else:
        print(f"WARNING: {file} not found — skipping")

if dep_dfs:
    df_departures = pd.concat(dep_dfs, ignore_index=True)

    # Parse departure datetime (Block Schedule with Off Scheduled fallback)
    df_departures['Scheduled_Departure_Datetime'] = pd.to_datetime(
        df_departures['Block Schedule'], errors='coerce'
    )
    mask = df_departures['Scheduled_Departure_Datetime'].isna()
    if 'Off Scheduled' in df_departures.columns:
        df_departures.loc[mask, 'Scheduled_Departure_Datetime'] = pd.to_datetime(
            df_departures.loc[mask, 'Off Scheduled'], errors='coerce'
        )
        print(f"Departure scheduled: {mask.sum():,} rows filled from 'Off Scheduled'")

    # Parse delay (canonical name after harmonization)
    if 'Total Calculated Delay' in df_departures.columns:
        df_departures['Dep_Calculated_Delay'] = df_departures['Total Calculated Delay']

    # Parse taxi time (canonical name after harmonization)
    if 'Total Taxi Time Calc' in df_departures.columns:
        df_departures['Dep_Taxi_Time'] = pd.to_numeric(
            df_departures['Total Taxi Time Calc'], errors='coerce'
        )

    # Sort and dedup
    df_departures = df_departures.sort_values('Scheduled_Departure_Datetime')
    before = len(df_departures)
    df_departures = df_departures.drop_duplicates(
        subset=['Registration', 'Call Sign', 'Block Schedule', 'Block Actual'],
        keep='first'
    ).reset_index(drop=True)

    df_departures['Dep_Date'] = df_departures['Scheduled_Departure_Datetime'].dt.date
    df_departures['Dep_Hour'] = df_departures['Scheduled_Departure_Datetime'].dt.hour

    print(f"\nDedup: {before:,} → {len(df_departures):,} ({before - len(df_departures):,} removed)")
    print(f"Total departures: {len(df_departures):,}")
    print(f"Date range: {df_departures['Dep_Date'].dropna().min()} to {df_departures['Dep_Date'].dropna().max()}")
else:
    df_departures = None
    print("WARNING: No departure data loaded")

Loaded LGA_Jan-May2025_Departures.csv: 67,605 rows
  → v2 schema: Carrier codes mapped (66,550 commercial, 1,055 GA/other, unmapped codes: ['JNY', 'Unknown', 'WMN', 'NJE', 'RVF'])
Loaded LGA_May-June2025_Departures.csv: 20,192 rows
  → Old schema, no renaming needed
Loaded LGA_July2025_Departures.csv: 15,960 rows
  → Old schema, no renaming needed
Loaded LGA_Aug-Sept2025_Departures.csv: 16,303 rows
  → Old schema, no renaming needed
Loaded LGA_Sept-Oct2025_Departures.csv: 30,044 rows
  → v2 schema: Carrier codes mapped (29,547 commercial, 497 GA/other, unmapped codes: ['Unknown', 'IJM', 'FXV', 'NJM', 'HER'])
Departure scheduled: 52,457 rows filled from 'Off Scheduled'

Dedup: 150,104 → 148,061 (2,043 removed)
Total departures: 148,061
Date range: 2024-12-31 to 2025-10-31


## 3. Missed Approaches (for lag features)

Missed approach events at LGA — used for `missed_approach_1h` lag feature. Expanded to 3 files.

In [5]:
# Load missed approaches data (V9.0: 3 files)
ma_files = [
    'LGA_Jan-May2025_MissedApproaches.csv',       # new schema
    'LGA_Summer2025_MissedApproaches.csv',         # old schema
    'LGA_Sept-Oct2025_MissedApproaches.csv',       # new schema
]

ma_dfs = []
for file in ma_files:
    filepath = DATA_RAW / file
    if filepath.exists():
        df_temp = pd.read_csv(filepath)
        print(f"Loaded {file}: {len(df_temp)} rows")
        df_temp = harmonize_flight_columns(df_temp, direction='arrival')
        ma_dfs.append(df_temp)
    else:
        print(f"WARNING: {file} not found — skipping")

if ma_dfs:
    df_missed = pd.concat(ma_dfs, ignore_index=True)

    # Parse datetime — missed approaches use On Actual (no Block Schedule/Actual)
    if 'On Actual' in df_missed.columns:
        df_missed['MA_Datetime'] = pd.to_datetime(df_missed['On Actual'], errors='coerce')
    elif 'On Scheduled' in df_missed.columns:
        df_missed['MA_Datetime'] = pd.to_datetime(df_missed['On Scheduled'], errors='coerce')

    df_missed['MA_Date'] = df_missed['MA_Datetime'].dt.date
    df_missed['MA_Hour'] = df_missed['MA_Datetime'].dt.hour

    print(f"\nTotal missed approaches: {len(df_missed)}")
    print(f"Date range: {df_missed['MA_Date'].dropna().min()} to {df_missed['MA_Date'].dropna().max()}")
    print(f"Missing MA_Datetime: {df_missed['MA_Datetime'].isna().sum()}")
else:
    df_missed = None
    print("WARNING: No missed approach files found — missed_approach_1h will be 0")

Loaded LGA_Jan-May2025_MissedApproaches.csv: 568 rows
  → v1 schema: Airline Name renamed to Marketing Airline Desc
Loaded LGA_Summer2025_MissedApproaches.csv: 332 rows
  → Old schema, no renaming needed
Loaded LGA_Sept-Oct2025_MissedApproaches.csv: 151 rows
  → v1 schema: Airline Name renamed to Marketing Airline Desc

Total missed approaches: 1051
Date range: 2025-01-01 to 2025-10-31
Missing MA_Datetime: 332


## 4. Rejected Takeoffs

Rejected takeoff events at LGA. Expanded to 3 files (Jan-Oct 2025).

In [6]:
# Load rejected takeoffs data (V9.0: 3 files)
rt_files = [
    'LGA_Jan-May2025_RejectedTakeoffs.csv',       # new schema
    'LGA_Summer2025_RejectedTakeoffs.csv',         # old schema
    'LGA_Sept-Oct2025_RejectedTakeoffs.csv',       # new schema
]

rt_dfs = []
for file in rt_files:
    filepath = DATA_RAW / file
    if filepath.exists():
        df_temp = pd.read_csv(filepath)
        print(f"Loaded {file}: {len(df_temp)} rows")
        df_temp = harmonize_flight_columns(df_temp, direction='departure')
        rt_dfs.append(df_temp)
    else:
        print(f"WARNING: {file} not found — skipping")

if rt_dfs:
    df_rejected = pd.concat(rt_dfs, ignore_index=True)

    # Parse datetime — use Block Actual or Off Actual
    if 'Block Actual' in df_rejected.columns:
        df_rejected['RT_Datetime'] = pd.to_datetime(df_rejected['Block Actual'], errors='coerce')
    elif 'Off Actual' in df_rejected.columns:
        df_rejected['RT_Datetime'] = pd.to_datetime(df_rejected['Off Actual'], errors='coerce')
    
    # Fallback to scheduled
    mask = df_rejected['RT_Datetime'].isna() if 'RT_Datetime' in df_rejected.columns else pd.Series([True]*len(df_rejected))
    if mask.any() and 'Block Schedule' in df_rejected.columns:
        df_rejected.loc[mask, 'RT_Datetime'] = pd.to_datetime(
            df_rejected.loc[mask, 'Block Schedule'], errors='coerce'
        )

    df_rejected['RT_Date'] = df_rejected['RT_Datetime'].dt.date
    df_rejected['RT_Hour'] = df_rejected['RT_Datetime'].dt.hour

    print(f"\nTotal rejected takeoffs: {len(df_rejected)}")
    print(f"Date range: {df_rejected['RT_Date'].dropna().min()} to {df_rejected['RT_Date'].dropna().max()}")
    print(f"Missing RT_Datetime: {df_rejected['RT_Datetime'].isna().sum()}")
else:
    df_rejected = None
    print("WARNING: No rejected takeoff files found")

Loaded LGA_Jan-May2025_RejectedTakeoffs.csv: 57 rows
  → v1 schema: Airline Name renamed to Marketing Airline Desc
Loaded LGA_Summer2025_RejectedTakeoffs.csv: 96 rows
  → Old schema, no renaming needed
Loaded LGA_Sept-Oct2025_RejectedTakeoffs.csv: 41 rows
  → v1 schema: Airline Name renamed to Marketing Airline Desc

Total rejected takeoffs: 194
Date range: NaT to NaT
Missing RT_Datetime: 194


## 5. FAA Operational Data

Four FAA Traffic Management datasets (V9.0: replaced with expanded files):
- **GDP** (Ground Delay Program): 68 events (was ~15)
- **GSP** (Ground Stop): 272 events (was ~7) — **39x expansion**
- **DD** (Departure Delays): 693 events (was ~20) — **12x expansion**
- **AD** (Arrival Delays): 56 events

**Important**: New FAA CSVs have metadata rows at the end ("Applied filters:...") that must be stripped.

In [7]:
# Load FAA operational data — try multiple naming conventions

def load_faa_file(data_dir, name_candidates, label):
    """Try loading FAA CSV from multiple possible filenames."""
    for fname in name_candidates:
        fpath = data_dir / fname
        if fpath.exists():
            df_faa = pd.read_csv(fpath)
            print(f"  {label}: Loaded {fname} — {len(df_faa)} rows (raw)")
            return df_faa
    print(f"  {label}: NOT FOUND (tried {name_candidates}) — will skip in Notebook 02")
    return None

def strip_faa_metadata(df_faa):
    """Remove metadata/filter rows at the end of FAA CSVs.
    These rows contain text like 'Applied filters:', 'PA Airport Code is LGA', etc.
    """
    if df_faa is None or len(df_faa) == 0:
        return df_faa
    first_col = df_faa.columns[0]
    # Keep only rows where first column looks like a datetime (contains '/')
    mask = df_faa[first_col].astype(str).str.contains(r'\d{1,2}/\d{1,2}/\d{4}', na=False)
    removed = (~mask).sum()
    df_clean = df_faa[mask].reset_index(drop=True)
    if removed > 0:
        print(f"    Stripped {removed} metadata rows")
    return df_clean

print("Loading FAA operational data (V9.0 expanded)...")

# GDP (Ground Delay Program) — has Start/End Time
df_gdp = load_faa_file(DATA_RAW, [
    'LGA_FAA_GroundDelays2025.csv',
    'LGA_FAA_GrounDelays2025.csv',   # known typo
    'df_FAA_GroundDelays2025.csv',
], 'GDP')
df_gdp = strip_faa_metadata(df_gdp)

# GSP (Ground Stop Program) — has Start/End Time
df_gsp = load_faa_file(DATA_RAW, [
    'LGA_FAA_GroundStops2025.csv',
    'df_FAA_GroundStops2025.csv',
], 'GSP')
df_gsp = strip_faa_metadata(df_gsp)

# DD (Departure Delays) — only Update Time
df_dd = load_faa_file(DATA_RAW, [
    'LGA_FAA_DepartureDelays2025.csv',
    'df_FAA_DepartureDelays2025.csv',
], 'DD')
df_dd = strip_faa_metadata(df_dd)
# Harmonize column name: 'DD Reason Description' → 'DD Reason' (NB02 compatibility)
if df_dd is not None and 'DD Reason Description' in df_dd.columns:
    df_dd = df_dd.rename(columns={'DD Reason Description': 'DD Reason'})
    print("    Renamed 'DD Reason Description' → 'DD Reason'")

# AD (Arrival Delays) — only Update Time
df_ad = load_faa_file(DATA_RAW, [
    'LGA_FAA_ArrivalDelays2025.csv',
    'df_FAA_ArrivalDelays2025.csv',
], 'AD')
df_ad = strip_faa_metadata(df_ad)

# Parse datetime columns for each FAA dataset
if df_gdp is not None:
    df_gdp['GDP_Start'] = pd.to_datetime(df_gdp['GDP Start Time'], errors='coerce')
    df_gdp['GDP_End'] = pd.to_datetime(df_gdp['GDP End Time'], errors='coerce')
    df_gdp['GDP_Cancelled'] = pd.to_datetime(df_gdp.get('GDP Cancelled Time'), errors='coerce')
    df_gdp['GDP_Effective_End'] = df_gdp['GDP_Cancelled'].fillna(df_gdp['GDP_End'])
    df_gdp['GDP_Avg_Delay_Min'] = pd.to_numeric(df_gdp['GDP Avg Delay'], errors='coerce')
    print(f"  GDP date range: {df_gdp['GDP_Start'].min()} to {df_gdp['GDP_Start'].max()}")

if df_gsp is not None:
    df_gsp['GSP_Start'] = pd.to_datetime(df_gsp['GSP Start Time'], errors='coerce')
    df_gsp['GSP_End'] = pd.to_datetime(df_gsp['GSP End Time'], errors='coerce')
    df_gsp['GSP_Cancelled'] = pd.to_datetime(df_gsp.get('GSP Cancelled Time'), errors='coerce')
    df_gsp['GSP_Effective_End'] = df_gsp['GSP_Cancelled'].fillna(df_gsp['GSP_End'])
    print(f"  GSP date range: {df_gsp['GSP_Start'].min()} to {df_gsp['GSP_Start'].max()}")

if df_dd is not None:
    df_dd['DD_Update'] = pd.to_datetime(df_dd['DD Update Time'], errors='coerce')
    df_dd['DD_Avg_Delay_Min'] = pd.to_numeric(df_dd['DD Avg Delay'], errors='coerce')
    print(f"  DD date range: {df_dd['DD_Update'].min()} to {df_dd['DD_Update'].max()}")

if df_ad is not None:
    df_ad['AD_Update'] = pd.to_datetime(df_ad['AD Update Time'], errors='coerce')
    df_ad['AD_Avg_Delay_Min'] = pd.to_numeric(df_ad['AD Avg Delay'], errors='coerce')
    print(f"  AD date range: {df_ad['AD_Update'].min()} to {df_ad['AD_Update'].max()}")

# Summary
faa_counts = {
    'GDP': len(df_gdp) if df_gdp is not None else 0,
    'GSP': len(df_gsp) if df_gsp is not None else 0,
    'DD': len(df_dd) if df_dd is not None else 0,
    'AD': len(df_ad) if df_ad is not None else 0,
}
print(f"\nFAA data summary: {faa_counts}")
print(f"Total FAA events: {sum(faa_counts.values())}")

Loading FAA operational data (V9.0 expanded)...
  GDP: Loaded LGA_FAA_GrounDelays2025.csv — 69 rows (raw)
    Stripped 2 metadata rows
  GSP: Loaded LGA_FAA_GroundStops2025.csv — 273 rows (raw)
    Stripped 2 metadata rows
  DD: Loaded LGA_FAA_DepartureDelays2025.csv — 694 rows (raw)
    Stripped 2 metadata rows
    Renamed 'DD Reason Description' → 'DD Reason'
  AD: Loaded LGA_FAA_ArrivalDelays2025.csv — 57 rows (raw)
    Stripped 2 metadata rows
  GDP date range: 2025-01-06 13:00:00 to 2025-11-04 09:59:00
  GSP date range: 2025-01-07 17:00:00 to 2025-11-04 09:06:00
  DD date range: 2025-01-06 16:32:00 to 2025-11-04 10:32:00
  AD date range: 2025-01-19 19:05:00 to 2025-10-28 15:00:00

FAA data summary: {'GDP': 67, 'GSP': 271, 'DD': 692, 'AD': 55}
Total FAA events: 1085


## 6. Data Exploration & Cleaning

In [8]:
# Basic info
print("Data Info:")
df.info()

print("\n" + "="*50)
print("Missing Values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148165 entries, 0 to 148164
Data columns (total 21 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   Date                                 148165 non-null  object 
 1   Registration                         148165 non-null  object 
 2   Call Sign                            148165 non-null  object 
 3   Flight Direction                     148165 non-null  object 
 4   Carrier                              97676 non-null   object 
 5   Non-PA Airport                       148165 non-null  object 
 6   Body Type Desc                       50489 non-null   object 
 7   Gate                                 148165 non-null  object 
 8   Terminal Code                        148165 non-null  object 
 9   Arrival Runway                       148165 non-null  object 
 10  Block Schedule                       145689 non-null  object 
 11  Bl

In [9]:
# Parse datetime columns using actual CSV column names
# Original columns: 'Block Schedule' (datetime), 'Block Actual' (datetime)
# Fallback: 'On Scheduled', 'On Actual' for flights missing Block times (e.g., Air Canada Commuter)

# Create datetime columns from Block Schedule/Actual with fallback
# V9.0: format='mixed' handles mixed formats across old/new CSV schemas
df['Scheduled_Arrival_Datetime'] = pd.to_datetime(df['Block Schedule'], format='mixed')
mask_sched = df['Scheduled_Arrival_Datetime'].isna()
if 'On Scheduled' in df.columns:
    df.loc[mask_sched, 'Scheduled_Arrival_Datetime'] = pd.to_datetime(df.loc[mask_sched, 'On Scheduled'], format='mixed')
    print(f"Scheduled: {mask_sched.sum()} rows filled from 'On Scheduled'")

df['Actual_Arrival_Datetime'] = pd.to_datetime(df['Block Actual'], format='mixed')
mask_actual = df['Actual_Arrival_Datetime'].isna()
if 'On Actual' in df.columns:
    df.loc[mask_actual, 'Actual_Arrival_Datetime'] = pd.to_datetime(df.loc[mask_actual, 'On Actual'], format='mixed')
    print(f"Actual: {mask_actual.sum()} rows filled from 'On Actual'")

# Check remaining missing
print(f"\nRemaining missing - Scheduled: {df['Scheduled_Arrival_Datetime'].isna().sum()}, Actual: {df['Actual_Arrival_Datetime'].isna().sum()}")

# Extract date components
df['Date'] = df['Scheduled_Arrival_Datetime'].dt.date
df['Hour'] = df['Scheduled_Arrival_Datetime'].dt.hour
df['DayOfWeek'] = df['Scheduled_Arrival_Datetime'].dt.dayofweek
df['Month'] = df['Scheduled_Arrival_Datetime'].dt.month
df['IsWeekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)

print("\nDatetime features created.")
print(f"Date range: {df['Date'].dropna().min()} to {df['Date'].dropna().max()}")

Scheduled: 2476 rows filled from 'On Scheduled'
Actual: 3265 rows filled from 'On Actual'

Remaining missing - Scheduled: 1006, Actual: 623

Datetime features created.
Date range: 2024-12-31 to 2025-10-31


In [10]:
# Use existing delay column from raw data (or calculate if needed)
# Original column: 'Total Calculated Delay' (in minutes)

if 'Total Calculated Delay' in df.columns:
    # Use existing column, rename for consistency
    df['Total_Calculated_Delay'] = df['Total Calculated Delay']
    print("Using existing 'Total Calculated Delay' column from raw data.")
else:
    # Calculate if not present
    df['Total_Calculated_Delay'] = (
        df['Actual_Arrival_Datetime'] - df['Scheduled_Arrival_Datetime']
    ).dt.total_seconds() / 60
    print("Calculated delay from datetime columns.")

print(f"\nDelay statistics:")
print(df['Total_Calculated_Delay'].describe())

Using existing 'Total Calculated Delay' column from raw data.

Delay statistics:
count    148165.000000
mean         11.531981
std          66.815855
min        -792.000000
25%         -16.000000
50%          -6.000000
75%          13.000000
max        1615.000000
Name: Total_Calculated_Delay, dtype: float64


## 7. Target Variable (Is_Delayed)

In [11]:
# DOT standard: Delayed if > 15 minutes late
df['Is_Delayed'] = (df['Total_Calculated_Delay'] > 15).astype(int)

print("Target Variable Distribution:")
print(df['Is_Delayed'].value_counts())
print(f"\nDelay Rate: {df['Is_Delayed'].mean()*100:.2f}%")

Target Variable Distribution:
Is_Delayed
0    113856
1     34309
Name: count, dtype: int64

Delay Rate: 23.16%


In [12]:
# Clean terminal column (actual column name: 'Terminal Code')
if 'Terminal Code' in df.columns:
    df['Terminal_Clean'] = df['Terminal Code'].fillna('Unknown')
elif 'Terminal' in df.columns:
    df['Terminal_Clean'] = df['Terminal'].fillna('Unknown')

# Clean runway column (actual column name: 'Arrival Runway')
if 'Arrival Runway' in df.columns:
    df['Runway_Clean'] = df['Arrival Runway'].fillna('Unknown')
elif 'Runway' in df.columns:
    df['Runway_Clean'] = df['Runway'].fillna('Unknown')

print(f"Terminal values: {df['Terminal_Clean'].unique()[:10] if 'Terminal_Clean' in df.columns else 'N/A'}")
print(f"Runway values: {df['Runway_Clean'].unique()[:10] if 'Runway_Clean' in df.columns else 'N/A'}")

Terminal values: ['Terminal B' 'Terminal A' 'Terminal C' 'LGA-Unknown' 'Unknown']
Runway values: [4 22 31 13 '22' '31' '22-22-31-31' '4' '22-31' '22-22']


## 8. Save Processed Data

In [13]:
# Create processed directory if not exists
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

def fix_mixed_types_for_parquet(dataframe):
    """Fix mixed int/str columns that cause ArrowInvalid on parquet save."""
    for col in dataframe.columns:
        if dataframe[col].dtype == object:
            dataframe[col] = dataframe[col].astype(str).replace({'nan': pd.NA, 'None': pd.NA})
    return dataframe

# Save arrivals
df = fix_mixed_types_for_parquet(df)
output_path = DATA_PROCESSED / 'arrivals_processed.parquet'
df.to_parquet(output_path, index=False)
print(f"Saved arrivals: {output_path} ({df.shape})")

# Save departures
if df_departures is not None:
    df_departures = fix_mixed_types_for_parquet(df_departures)
    dep_path = DATA_PROCESSED / 'departures_processed.parquet'
    df_departures.to_parquet(dep_path, index=False)
    print(f"Saved departures: {dep_path} ({df_departures.shape})")

# Save missed approaches
if df_missed is not None:
    df_missed = fix_mixed_types_for_parquet(df_missed)
    ma_path = DATA_PROCESSED / 'missed_approaches_processed.parquet'
    df_missed.to_parquet(ma_path, index=False)
    print(f"Saved missed approaches: {ma_path} ({df_missed.shape})")

# Save rejected takeoffs
if df_rejected is not None:
    df_rejected = fix_mixed_types_for_parquet(df_rejected)
    rt_path = DATA_PROCESSED / 'rejected_takeoffs_processed.parquet'
    df_rejected.to_parquet(rt_path, index=False)
    print(f"Saved rejected takeoffs: {rt_path} ({df_rejected.shape})")

# Save FAA datasets
for name, df_faa in [('gdp', df_gdp), ('gsp', df_gsp), ('dd', df_dd), ('ad', df_ad)]:
    if df_faa is not None:
        df_faa = fix_mixed_types_for_parquet(df_faa)
        faa_path = DATA_PROCESSED / f'faa_{name}_processed.parquet'
        df_faa.to_parquet(faa_path, index=False)
        print(f"Saved FAA {name.upper()}: {faa_path} ({df_faa.shape})")

print("\nAll processed files saved.")

Saved arrivals: ../../../data/processed/arrivals_processed.parquet ((148165, 31))
Saved departures: ../../../data/processed/departures_processed.parquet ((148061, 26))
Saved missed approaches: ../../../data/processed/missed_approaches_processed.parquet ((1051, 23))
Saved rejected takeoffs: ../../../data/processed/rejected_takeoffs_processed.parquet ((194, 23))
Saved FAA GDP: ../../../data/processed/faa_gdp_processed.parquet ((67, 14))
Saved FAA GSP: ../../../data/processed/faa_gsp_processed.parquet ((271, 12))
Saved FAA DD: ../../../data/processed/faa_dd_processed.parquet ((692, 10))
Saved FAA AD: ../../../data/processed/faa_ad_processed.parquet ((55, 10))

All processed files saved.
